`r format(Sys.time(), '🗓️ %d %B, %Y 🕔 %H:%M')`

In [ ]:
library(dplyr)
library(DT)
library(highcharter)
library(plotly)
library(scales)

In [ ]:
file <- params$blockfile
df <- read.table(
  file, header = T,
  colClasses = c("factor","factor","integer","integer", "integer")
)
if(nrow(df) < 1){
  cat(paste("No phase blocks were observed in the input file", file, "\n"))
  knitr::knit_exit()
}
df$pos_end <- df$pos_start + df$block_length
df <- df[df$block_length > 0, c(1,2,3,4,6,5)]
levels(df$sample) <- basename(levels(df$sample))

In [ ]:
NX <- function(lengths, X=50){
  lengths <- as.numeric(sort(lengths, decreasing=TRUE))
  index <- which(cumsum(lengths) / sum(lengths) >= X/100)[1]
  return(lengths[index])
}

# General Stats
## fileheader 
<h1> Haplotype Phasing Results </h1>
The data presented here are the results of phasing genotypes into haplotypes using
[HapCut2](https://github.com/vibansal/HapCUT2). The information is derived from `r file`,
which summarized information across all samples using the `.blocks` files HapCut2
generates. The VCF files HapCut2 also generates were not used as they lack
some of the information in the `.blocks` files that were collated in this report.
This page shows general and per-contig information. The `Per-Sample Stats` tab in
the navigation bar above will show you statistics relating to haplotypes on the
per-sample level. Haplotype blocks with a size of `0` were removed from the data.

In [ ]:
overall <- df %>% summarise(
    total_snps = sum(n_snp),
    mean_snps = round(mean(n_snp), digits = 0),
    median_snps = median(n_snp),
    max_blocksize = max(block_length),
    N50 = NX(block_length, 50),
    N75 = NX(block_length, 75),
    N90 = NX(block_length, 90)
)

##

In [ ]:
list(
  color = "light",
  value = prettyNum(overall$total_snps[1], big.mark = ",")
)

In [ ]:
list(
  color = "#dfdfdf",
  value = paste0(prettyNum(overall$max_blocksize[1], big.mark = ","), "bp")
)

In [ ]:
list(
  color = "#dfdfdf",
  value = prettyNum(overall$mean_snps[1], big.mark = ",")
)

In [ ]:
list(
  color = "#dfdfdf",
  value = prettyNum(overall$median_snps[1], big.mark = ",")
)

In [ ]:
list(
  color = "info",
  value = paste0(prettyNum(overall$N50[1], big.mark = ","), "bp")
)

In [ ]:
list(
  color = "info",
  value = paste0(prettyNum(overall$N75[1], big.mark = ","), "bp")
)

In [ ]:
list(
  color = "info",
  value = paste0(prettyNum(overall$N90[1], big.mark = ","), "bp")
)

## distribution plots
::: {.card title="Haplotype Length Distribution" expandable="true"}
This shows the distribution of haplotype length (in base pairs). 
Phasing typically has a few very large haplotypes, resulting in extreme 
right-tails in these distributions, therefore, the X-axis is **log-scaled lengths**
to collapse the right tail for better readability. 

In [ ]:
hs <- hist(
  df$block_length,
  breaks = 500,
  plot = F
)
hs <- data.frame(val = hs$mids, freq = hs$counts)
hchart(hs, "areaspline", hcaes(x = val, y = freq), color = "#7eb495", name = "count") |>
  hc_xAxis(title = list(text = "haplotype length in base pairs (log scale)"), type = "logarithmic") |>
  hc_title(text = "Haplotype block lengths") |>
  hc_yAxis(title = list(text = "count")) |>
  hc_tooltip(crosshairs = TRUE) |>
  hc_exporting(enabled = T, filename = "haplotype.hist",
    buttons = list(contextButton = list(menuItems = c("downloadPNG", "downloadJPEG", "downloadPDF", "downloadSVG")))
  )

:::

###
::: {.card title="NX Information" expandable="true"}

An **NX** metric (e.g. **N50**) is the length of the shortest molecule in the group of longest molecules that together
represent at least **X%** of the total molecules by length. For example, `N50` would be the shortest molecule in the 
group of longest molecules that together represent **50%** of the total molecules by length (sort of like a median).

In [ ]:
nx_df <- df %>% group_by(sample, contig) %>%
  summarize(
    N50 = NX(block_length, 50),
    N75 = NX(block_length, 75),
    N90 = NX(block_length, 90)
  )

In [ ]:
if(nrow(nx_df) > 1){
  highchart() |>
    hc_chart(type = "area", animation = F) |>
    hc_add_series(data = density(nx_df$N50), name = "N50", type = "areaspline") |>
    hc_add_series(data = density(nx_df$N75), name = "N75", type = "areaspline") |>
    hc_add_series(data = density(nx_df$N90), name = "N90", color = "#d3805f", type = "areaspline") |>
    hc_tooltip(enabled = FALSE) |>
    hc_title(text = "NX Stats Across Samples and Contigs") |>
    hc_xAxis(title = list(text = "NX value"), min = 0) |>
    hc_yAxis(title = list(text = "density")) |>
    hc_exporting(enabled = T, filename = "NX.stats",
      buttons = list(contextButton = list(menuItems = c("downloadPNG", "downloadJPEG", "downloadPDF", "downloadSVG")))
    )
 } else {
  cat("At least two samples are required to plot a distribution\n")
}

:::

##

In [ ]:
percontig <- df %>% group_by(contig) %>% summarise(
    n_haplo = round(length(n_snp)),
    mean_snps = round(mean(n_snp), digits = 0),
    median_snps = median(n_snp),
    mean_blocksize = round(mean(block_length), digits = 0),
    N50 = NX(block_length, 50),
    max_blocksize = max(block_length)
  )

In [ ]:
dropdown_buttons <- function(CONTIGS, CUTOFF = 0){
  buttonlist <- list()
  idx <- 0
  n_contigs <- length(CONTIGS)
  for(i in CONTIGS){
    idx <- idx + 1
    if((CUTOFF > 0) && (idx > CUTOFF)) break
    visibility <- as.list(rep(FALSE, n_contigs))
    visibility[[idx]] <- TRUE
    buttonlist[[idx]] <- list(method = "restyle", label = i, args = list("visible", visibility))
  }
  return( list(list(y = 1, buttons = buttonlist)) )
}

In [ ]:
plotting_contigs <- params$contigs
if (all(plotting_contigs == "default")){
  .contigs <- group_by(df, contig) %>%
    summarize(size = max(pos_end)) %>%
    arrange(desc(size))
  plot_contigs <- .contigs$contig[1:min(nrow(.contigs), 40)] 
  } else {
  plot_contigs <- plotting_contigs
}

::: {.card title="Haplotype Lengths Per Contig" expandable="true"}
This is the distribution of haplotype length (in base pairs) for 
each contig, up to 300 contigs. Phasing typically has a few very large haplotypes, resulting in extreme 
right-tails in these distributions, therefore, the haplotype lengths are **log-scaled lengths**
to collapse the right tail for better readability. The dotted vertical bar represents the mean haplotype length.

In [ ]:
fig <- plot_ly(hoverinfo = "none") %>%
  layout(
    xaxis = list(title = "Log-Scaled Haplotype Length (bp)", fixedrange = TRUE),#, type = "log"),
    yaxis = list(fixedrange = TRUE),
    title = "Haplotype Lengths by Contig",
    showlegend = F,
    updatemenus = dropdown_buttons(plot_contigs)
  ) %>% config(displayModeBar = FALSE)
# Loop over the categories
for (cont in plot_contigs) {
  subset_cont <- df$block_length[df$contig == cont]
  fig <- fig %>%
    add_trace(
      x = log(subset_cont),
      type = "violin",
      name = cont,
      side = "positive",
      meanline = list(visible = T),
      visible = "legendonly"
    )
}

fig

:::

###
:::{.card title="Haplotype Stats by Contig" expandable="true"}

In [ ]:
column_description <- c(
  "name of the contig",
  "total haplotypes on the contig",
  "mean number of SNPs per haplotype",
  "median number of SNPs per haplotype",
  "mean length of haplotypes, given in base pairs",
  "N50 length of the haplotype blocks, given in base pairs",
  "length of the largest haplotype, given in base pairs"
  )

headerCallback <- c(
  "function(thead, data, start, end, display){",
  paste0(" var tooltips = ['", paste(column_description, collapse = "','"), "'];"),
  "  for(var i=0; i<tooltips.length; i++){",
  "    $('th:eq('+i+')',thead).attr('title', tooltips[i]);",
  "  }",
  "}"
)

DT::datatable(
  percontig,
  rownames = F,
  extensions = 'Buttons',
  colnames = c("Contig", "Total Haplotypes", "Mean SNPs", "Median SNPs", "Mean Haplotype Length", "Haplotype N50", "Largest Haplotype"),
  fillContainer = T,
  options = list(
    dom = 'Brtp',
    buttons = list(list(extend = "csv",filename = "phasing_per_contig")), 
    scrollX = TRUE,
    headerCallback = JS(headerCallback)
  )
)

:::


# Per-Sample
##
The plots below shows the distribution of haplotype lengths (in base pairs) for each 
sample. While no Y-axis for these counts are provided, this plot is intended to be more of
a visual reference of the relative distributions of haplotype lengths among samples.
Phasing typically has a few very large haplotypes, resulting in extreme 
right-tails in these distributions, therefore, the haplotype lengths are
**log-scaled lengths** to collapse the right tail for better readability.
The dotted vertical bar represents the mean haplotype length. 

## per sample

In [ ]:
persample <- df %>% group_by(sample) %>% summarise(
    n_haplo = sum(length(n_snp)),
    mean_snps = round(mean(n_snp), digits = 0),
    median_snps = median(n_snp),
    mean_blocksize = round(mean(block_length), digits = 0),
    N50 = NX(block_length, 50),
    max_blocksize = max(block_length)
)

column_description <- c(
  "name of the sample",
  "number of haplotypes",
  "mean number of SNPs per haplotype",
  "median number of SNPs per haplotype",
  "mean length of haplotypes, given in base pairs",
  "N50 length of the haplotype blocks, given in base pairs",
  "length of the largest haplotype, given in base pairs"
)

headerCallback <- c(
  "function(thead, data, start, end, display){",
  paste0(" var tooltips = ['", paste(column_description, collapse = "','"), "'];"),
  "  for(var i=0; i<tooltips.length; i++){",
  "    $('th:eq('+i+')',thead).attr('title', tooltips[i]);",
  "  }",
  "}"
)

DT::datatable(
  persample,
  rownames = F,
  extensions = 'Buttons',
  colnames = c("Sample", "Haplotypes", "Mean SNPs", "Median SNPs", "Mean Haplotype Length", "Haplotype N50", "Largest Haplotype"),
  fillContainer = T,
  options = list(
    dom = 'Brtp', 
    buttons = list(list(extend = "csv",filename = "phasing_per_sample")), 
    scrollX = TRUE,
    headerCallback = JS(headerCallback)
  )
)

### the ridgeplot

In [ ]:
fig <- plot_ly(hoverinfo = "none") %>%
  layout(
    xaxis = list(title = "Log-Scaled Haplotype Length (bp)", fixedrange = TRUE),
    yaxis = list(fixedrange = TRUE),
    title = "Haplotype Lengths by Sample",
    showlegend = F,
    updatemenus = dropdown_buttons(levels(df$sample), 600)
  ) %>% config(displayModeBar = FALSE)
# Loop over the categories
.idx <- 0
for (samp in levels(df$sample)) {
  .idx <- .idx + 1
  if(.idx > 600) break
  subset_sample <- df$block_length[df$sample == samp] 
  fig <- fig %>%
    add_trace(
      x = log(subset_sample),
      type = "violin",
      name = samp,
      side = "positive",
      meanline = list(visible = T),
      visible = "legendonly"
    )
}

fig